In [ ]:
from pyspark.sql import functions as F

catalog = "cdac-project"
schema = "intelligent-main-folder"
metadata_table = f"`{catalog}`.`{schema}`.file_metadata"

In [ ]:
metadata_schema = (
    "file_id STRING, file_name STRING, file_path STRING, file_size LONG, "
    "file_extension STRING, document_type STRING, status STRING, "
    "ingestion_timestamp TIMESTAMP"
)

if spark.catalog.tableExists(metadata_table):
    metadata_df = spark.table(metadata_table)
else:
    # Table doesn't exist yet (first run) — start from an empty frame.
    metadata_df = spark.createDataFrame([], schema=metadata_schema)

display(metadata_df)

In [ ]:
new_files_df = metadata_df.filter(
    F.col("status") == "NEW"
)

display(new_files_df)

In [ ]:
input_path = "/Volumes/cdac-project/intelligent-main-folder/raw"

In [ ]:
files = dbutils.fs.ls(input_path)

# dbutils.fs.ls() also returns subdirectories (e.g. if someone
# organizes the raw folder into subfolders), not just files.
# Directory entries always have a name ending in "/" — without this
# filter, a subdirectory would get ingested as if it were an
# ingestible file, and Notebook 2 would fail trying to read it as a
# document.
files = [f for f in files if not f.name.endswith("/")]

files_df = (
    spark.createDataFrame(
        [(f.name, f.path, f.size) for f in files],
        ["file_name", "file_path", "file_size"]
    )
    .withColumn(
        "file_extension",
        F.lower(F.regexp_extract("file_name", r"\.([^.]+)$", 1))
    )
    .withColumn("document_type", F.lit("UNKNOWN"))
    .withColumn("status", F.lit("NEW"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn(
        "file_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("file_path"),
                F.col("file_size").cast("string")
            ),
            256
        )
    )
    .select(
        "file_id",
        "file_name",
        "file_path",
        "file_size",
        "file_extension",
        "document_type",
        "status",
        "ingestion_timestamp"
    )
)

display(files_df)

In [ ]:
existing_ids = {row.file_id for row in metadata_df.select("file_id").collect()}

# Only append files we haven't seen before — an overwrite here would reset
# the status of files already processed by later pipeline stages.
new_only_df = files_df.filter(~F.col("file_id").isin(existing_ids)) if existing_ids else files_df

new_only_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(metadata_table)

In [ ]:
display(spark.table(metadata_table))

In [ ]:
# ============================================================
# 6. PUBLISH WHETHER THERE'S ANYTHING NEW
# ============================================================
# The rest of the job (Notebooks 2-6) only need to run if this
# notebook actually found new files. This value is read by a
# Condition Task in the job — if false, Databricks skips every
# downstream task entirely (no environment attach, no %pip install,
# nothing), instead of each of those 5 notebooks starting up just to
# discover for itself that there's nothing new to do.

has_new_files = new_only_df.count() > 0
dbutils.jobs.taskValues.set(key="has_new_files", value=has_new_files)

print(f"has_new_files = {has_new_files}")